In [1]:
# importing setup
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt

# NBA data 
from nba_api.stats.endpoints import leaguegamelog, playergamelogs

# Models and Evaluators 
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error


In [2]:
# getting every players stats from the regular season.
log = leaguegamelog.LeagueGameLog(
      season="2025-26",
      season_type_all_star="Regular Season",
      player_or_team_abbreviation="P",
)

# getting the usg_pct from every player regular season
adv = playergamelogs.PlayerGameLogs(
    season_nullable="2025-26",
    season_type_nullable="Regular Season",
    measure_type_player_game_logs_nullable="Advanced"
)

# turning it into a table
players = log.get_data_frames()[0]
adv_df = adv.get_data_frames()[0]

# checking
print(players.shape)
print(players.head())

# save the file 
players.to_csv("../data/raw/player_logs_2025-26_regular.csv", index=False)
adv_df.to_csv("../data/raw/player_adv_2025-26_regular.csv", index=False)

(26651, 32)
  SEASON_ID  PLAYER_ID     PLAYER_NAME     TEAM_ID TEAM_ABBREVIATION  \
0     22025     203110  Draymond Green  1610612744               GSW   
1     22025    1627780  Gary Payton II  1610612744               GSW   
2     22025     203500    Steven Adams  1610612745               HOU   
3     22025    1630578  Alperen Sengun  1610612745               HOU   
4     22025    1631106      Tari Eason  1610612745               HOU   

               TEAM_NAME     GAME_ID   GAME_DATE    MATCHUP WL  ...  REB  AST  \
0  Golden State Warriors  0022500002  2025-10-21  GSW @ LAL  W  ...    7    9   
1  Golden State Warriors  0022500002  2025-10-21  GSW @ LAL  W  ...    2    1   
2        Houston Rockets  0022500001  2025-10-21  HOU @ OKC  L  ...   13    3   
3        Houston Rockets  0022500001  2025-10-21  HOU @ OKC  L  ...   11    7   
4        Houston Rockets  0022500001  2025-10-21  HOU @ OKC  L  ...    6    0   

   STL  BLK  TOV  PF  PTS  PLUS_MINUS  FANTASY_PTS  VIDEO_AVAILABLE 

In [3]:
# converting FGA into fantasy points 
scoring = {
    "FGA": -1,
    "FGM": 2,
    "PTS": 1,
    "FG3M": 1,
    "REB": 1,
    "AST": 2,
    "STL": 4,
    "BLK": 4,
    "FTM": 1,
    "TOV": -2,
}

# the stats we will make a model for
targets = list(scoring.keys())
print(targets)



['FGA', 'FGM', 'PTS', 'FG3M', 'REB', 'AST', 'STL', 'BLK', 'FTM', 'TOV']


In [4]:
# keeping necessary columns
helper_cols = ["PLAYER_ID", "PLAYER_NAME", "GAME_ID", "TEAM_ID", "GAME_DATE", "MATCHUP", "MIN", "FTA"]

keep = helper_cols + targets

fga_data = players[keep].copy()



In [5]:
# feature engineering rest days

# turn text into dates
fga_data["GAME_DATE"] = pd.to_datetime(fga_data["GAME_DATE"])

# putting each players game in order
fga_data = fga_data.sort_values(["PLAYER_ID", "GAME_DATE"])

# days since previous game 
fga_data["REST_DAYS"] = fga_data.groupby("PLAYER_ID")["GAME_DATE"].diff().dt.days

# fill first games and cap long gaps at 4
fga_data["REST_DAYS"] = fga_data["REST_DAYS"].fillna(4).clip(upper=4)

# flag back to back games 
fga_data["BACK_TO_BACK"] = (fga_data["REST_DAYS"] == 1).astype(int)

# check 
check = fga_data[fga_data["PLAYER_NAME"] == "Scottie Barnes"]
print(check[["GAME_DATE", "REST_DAYS"]].head(10))

      GAME_DATE  REST_DAYS
189  2025-10-22        4.0
394  2025-10-24        2.0
853  2025-10-26        2.0
1169 2025-10-27        1.0
1507 2025-10-29        2.0
1700 2025-10-31        2.0
1957 2025-11-02        2.0
2358 2025-11-04        2.0
2945 2025-11-07        3.0
3070 2025-11-08        1.0


In [6]:
# feature engineering home vs away games

fga_data["HOME"] = (~fga_data["MATCHUP"].str.contains("@")).astype(int)

print(fga_data[["MATCHUP", "HOME"]].head(10))
print(fga_data["HOME"].value_counts())

          MATCHUP  HOME
4711  LAL vs. UTA     1
5492    LAL @ UTA     0
5754  LAL vs. LAC     1
6094  LAL vs. DAL     1
6706  LAL vs. PHX     1
7129    LAL @ TOR     0
7747    LAL @ PHI     0
7920  LAL vs. SAS     1
8255    LAL @ PHX     0
8521    LAL @ UTA     0
HOME
0    13328
1    13323
Name: count, dtype: int64


In [7]:
# feature engineering rolling averages last 5 games for every target stat   # CHANGED

# make sure data is ordered 
fga_data = fga_data.sort_values(["PLAYER_ID", "GAME_DATE"])

# average each target stat over the last 5 games   # CHANGED: loop over targets instead of FGA only
for col in targets:
    fga_data[col + "_LAST5"] = (
        fga_data.groupby("PLAYER_ID")[col].transform(lambda s: s.shift(1).rolling(5).mean())
    )
    fga_data[col + "_LAST10"] = (
        fga_data.groupby("PLAYER_ID")[col].transform(lambda s: s.shift(1).rolling(10).mean())
    )
    fga_data[col + "_SEASONAVG"] = (
        fga_data.groupby("PLAYER_ID")[col].transform(lambda s: s.shift(1).expanding().mean())
    )
# check 
check = fga_data[fga_data["PLAYER_NAME"] == "Scottie Barnes"]
print(check[["GAME_DATE", "FGA", "FGA_LAST5", "REB", "REB_LAST5", "FGA_LAST10"]].head(12))   # CHANGED


      GAME_DATE  FGA  FGA_LAST5  REB  REB_LAST5  FGA_LAST10
189  2025-10-22   14        NaN    6        NaN         NaN
394  2025-10-24   15        NaN    5        NaN         NaN
853  2025-10-26   24        NaN   11        NaN         NaN
1169 2025-10-27    7        NaN    3        NaN         NaN
1507 2025-10-29   18        NaN    5        NaN         NaN
1700 2025-10-31   14       15.6   10        6.0         NaN
1957 2025-11-02   19       15.6   12        6.8         NaN
2358 2025-11-04   16       16.4    3        8.2         NaN
2945 2025-11-07   16       14.8   10        6.6         NaN
3070 2025-11-08   14       16.6    5        8.0         NaN
3545 2025-11-11    9       15.8   11        8.0        15.7
3965 2025-11-13   22       14.8   10        8.2        15.2


In [8]:
# feature engineering minutes last 5 games 

# make sure data is ordered 
fga_data = fga_data.sort_values(["PLAYER_ID", "GAME_DATE"])

#average the mins last 5 games 
fga_data["MIN_LAST5"] = (
    fga_data.groupby("PLAYER_ID")["MIN"].transform(lambda s: s.shift(1).rolling(5).mean())
)

# check 
check = fga_data[fga_data["PLAYER_NAME"] == "Scottie Barnes"]
print(check[["GAME_DATE", "MIN", "MIN_LAST5"]].head(10))

      GAME_DATE  MIN  MIN_LAST5
189  2025-10-22   32        NaN
394  2025-10-24   36        NaN
853  2025-10-26   34        NaN
1169 2025-10-27   30        NaN
1507 2025-10-29   33        NaN
1700 2025-10-31   35       33.0
1957 2025-11-02   32       33.6
2358 2025-11-04   28       32.8
2945 2025-11-07   35       31.6
3070 2025-11-08   32       32.6


In [9]:
# feature engineering rolling free throw attempts

# make sure data is ordered 
fga_data = fga_data.sort_values(["PLAYER_ID", "GAME_DATE"])

# average the last 5 games
fga_data["FTA_LAST5"] = (
    fga_data.groupby("PLAYER_ID")["FTA"].transform(lambda s: s.shift(1).rolling(5).mean())
)

# check 
check = fga_data[fga_data["PLAYER_NAME"] == "Scottie Barnes"]
print(check[["GAME_DATE", "FTA", "FTA_LAST5"]].head(10))

      GAME_DATE  FTA  FTA_LAST5
189  2025-10-22    4        NaN
394  2025-10-24    4        NaN
853  2025-10-26    5        NaN
1169 2025-10-27    1        NaN
1507 2025-10-29    6        NaN
1700 2025-10-31    4        4.0
1957 2025-11-02    0        4.0
2358 2025-11-04    2        3.2
2945 2025-11-07    4        2.6
3070 2025-11-08    3        3.2


In [10]:
# merging usg_pct into fga_data

# keep only matching column and new stat 
usage = adv_df[["PLAYER_ID", "GAME_ID", "USG_PCT"]].copy()

# make gameids into the same data type 
fga_data["GAME_ID"] = fga_data["GAME_ID"].astype(int)
usage["GAME_ID"] = usage["GAME_ID"].astype(int)

# add the usg_pct to each row matching the player and game
fga_data = fga_data.merge(usage, on=["PLAYER_ID", "GAME_ID"], how="left")

# check 
print(fga_data.shape)
print(fga_data["USG_PCT"].isna().sum())

(26651, 54)
0


In [11]:
# feature engineering field goal % over the last 5 games

# the sum of the previous FGA
fga_sum = fga_data.groupby("PLAYER_ID")["FGA"].transform(lambda s: s.shift(1).rolling(5).sum())

# the sum of the previous FGM
fgm_sum = fga_data.groupby("PLAYER_ID")["FGM"].transform(lambda s: s.shift(1).rolling(5).sum())

# FG PCT
fga_data["FG_PCT_LAST5"] = fgm_sum / fga_sum

# check 
check = fga_data[fga_data["PLAYER_NAME"] == "Scottie Barnes"]
print(check[["GAME_DATE", "FGM", "FGA", "FG_PCT_LAST5"]].head(10))

       GAME_DATE  FGM  FGA  FG_PCT_LAST5
13213 2025-10-22    9   14           NaN
13214 2025-10-24    6   15           NaN
13215 2025-10-26   13   24           NaN
13216 2025-10-27    3    7           NaN
13217 2025-10-29   11   18           NaN
13218 2025-10-31    6   14      0.538462
13219 2025-11-02    8   19      0.500000
13220 2025-11-04    9   16      0.500000
13221 2025-11-07    5   16      0.500000
13222 2025-11-08    7   14      0.469880


In [12]:
# feature engineering the USG_PCT

# make sure data is ordered 
fga_data = fga_data.sort_values(["PLAYER_ID", "GAME_DATE"])

# usage x minutes for each game 
fga_data["USG_X_MIN"] = fga_data["USG_PCT"] * fga_data["MIN"]

# adding all the USG_X_MIN per player 
usg_sum = fga_data.groupby("PLAYER_ID")["USG_X_MIN"].transform(lambda s: s.shift(1).rolling(5).sum())

# adding all the MIN per player
min_sum = fga_data.groupby("PLAYER_ID")["MIN"].transform(lambda s: s.shift(1).rolling(5).sum())

# getting the rolling 5 
fga_data["USG_PCT_LAST5"] = usg_sum / min_sum

#check 
check = fga_data[fga_data["PLAYER_NAME"] == "Scottie Barnes"]
print(check[["GAME_DATE", "MIN", "USG_PCT", "USG_PCT_LAST5"]].head(10))

       GAME_DATE  MIN  USG_PCT  USG_PCT_LAST5
13213 2025-10-22   32    0.191            NaN
13214 2025-10-24   36    0.230            NaN
13215 2025-10-26   34    0.337            NaN
13216 2025-10-27   30    0.136            NaN
13217 2025-10-29   33    0.296            NaN
13218 2025-10-31   35    0.238       0.240594
13219 2025-11-02   32    0.266       0.249500
13220 2025-11-04   28    0.254       0.257000
13221 2025-11-07   35    0.202       0.239253
13222 2025-11-08   32    0.221       0.250258


In [13]:

# split data by date 
train = fga_data[fga_data["GAME_DATE"] < "2026-02-01"]
val = fga_data[(fga_data["GAME_DATE"] >= "2026-02-01") & (fga_data["GAME_DATE"] < "2026-03-15")]
test = fga_data[fga_data["GAME_DATE"] >= "2026-03-15"]

# CHANGED: shared features + the last 5 average of every target stat
features_cols = (["HOME", "REST_DAYS", "BACK_TO_BACK", "MIN_LAST5", "FTA_LAST5", "USG_PCT_LAST5", "FG_PCT_LAST5"]
                 + [t + "_LAST5" for t in targets]
                 + [t + "_LAST10" for t in targets]
                 + [t + "_SEASONAVG" for t in targets])

# drop rows with missing rolling features (each player's first 10 games)
train = train.dropna(subset=features_cols)
val = val.dropna(subset=features_cols)
test = test.dropna(subset=features_cols)

# TimeSeriesSplit need to train rows in date order 
train = train.sort_values("GAME_DATE")

# CHANGED: only the features here, the target (y) is picked inside the function
X_train = train[features_cols]
X_val = val[features_cols]

# models 
models = {
    "linear": (make_pipeline(StandardScaler(), LinearRegression()), {}),
    "ridge": (make_pipeline(StandardScaler(), Ridge()), 
              {"ridge__alpha": [0.1, 1, 10, 100]}),
    "random_forest": (RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1),
                      {"min_samples_leaf": [20, 50 ,100], "max_features": [0.5, 0.8]}),
    "gradient_boosting": (HistGradientBoostingRegressor(random_state=42), 
                          {"learning_rate": [0.03, 0.1], "min_samples_leaf": [20, 50, 100]}),
}

tscv = TimeSeriesSplit(n_splits=4)
best_models = {}


# NEW: the same comparison as before, but for any target stat
def evaluate_target(target):
    y_train = train[target]   # CHANGED: was train["FGA"]
    y_val = val[target]       # CHANGED: was val["FGA"]

    # baseline = the stat's own last 5 average   # CHANGED: was "FGA_LAST5"
    baseline_col = target + "_LAST5"
    results = [{"target": target, "model": "baseline",
                "train_MAE": mean_absolute_error(y_train, train[baseline_col]), 
                "val_MAE": mean_absolute_error(y_val, val[baseline_col])}]

    # train each model and score MAE on train and validation 
    for name, (model, grid) in models.items():
        search = GridSearchCV(model, grid, cv=tscv, scoring="neg_mean_absolute_error")
        search.fit(X_train, y_train)
        model  = search.best_estimator_
        best_models[(target, name)] = model   # CHANGED: key is (stat, model name)

        train_preds = model.predict(X_train)
        preds = model.predict(X_val)

        train_mae = mean_absolute_error(y_train, train_preds)
        val_mae = mean_absolute_error(y_val, preds)

        print(f"{target} | {name}: {search.best_params_}")   # CHANGED: shows the stat too
        print(f"   train MAE = {train_mae:.3f} | val MAE = {val_mae:.3f} | gap = {val_mae - train_mae:.3f}")

        results.append({"target": target, "model": name, "train_MAE": train_mae, "val_MAE": val_mae})

    return results


In [16]:
# do the new features help? 

# version 1 features: shared features + last 5 only 
old_cols = (["HOME", "REST_DAYS", "BACK_TO_BACK", "MIN_LAST5", "FTA_LAST5", "USG_PCT_LAST5", "FG_PCT_LAST5"] + [t + "_LAST5" for t in targets])

# run with the OLD features 
X_train = train[old_cols]
X_val = val[old_cols]
results_old = pd.DataFrame(evaluate_target("PTS"))

# run with new features
X_train = train[features_cols]
X_val = val[features_cols]
results_new = pd.DataFrame(evaluate_target("PTS"))

# compare val MAE side by side
print(results_old[["model", "val_MAE"]])
print(results_new[["model", "val_MAE"]])

PTS | linear: {}
   train MAE = 4.696 | val MAE = 4.722 | gap = 0.025
PTS | ridge: {'ridge__alpha': 10}
   train MAE = 4.697 | val MAE = 4.722 | gap = 0.025
PTS | random_forest: {'max_features': 0.5, 'min_samples_leaf': 50}
   train MAE = 4.509 | val MAE = 4.755 | gap = 0.247
PTS | gradient_boosting: {'learning_rate': 0.03, 'min_samples_leaf': 100}
   train MAE = 4.524 | val MAE = 4.779 | gap = 0.255
PTS | linear: {}
   train MAE = 4.557 | val MAE = 4.650 | gap = 0.093
PTS | ridge: {'ridge__alpha': 100}
   train MAE = 4.558 | val MAE = 4.644 | gap = 0.086
PTS | random_forest: {'max_features': 0.5, 'min_samples_leaf': 50}
   train MAE = 4.320 | val MAE = 4.673 | gap = 0.352
PTS | gradient_boosting: {'learning_rate': 0.03, 'min_samples_leaf': 100}
   train MAE = 4.356 | val MAE = 4.688 | gap = 0.332
               model   val_MAE
0           baseline  4.841004
1             linear  4.721563
2              ridge  4.721853
3      random_forest  4.755464
4  gradient_boosting  4.779150
     

In [14]:
# NEW: run the comparison for each target stat
# CHANGED: run every stat in scoring (takes about 8 min)
run_targets = targets

all_results = []
for target in run_targets:
    all_results += evaluate_target(target)

results_df = pd.DataFrame(all_results)
results_df["gap"] = results_df["val_MAE"] - results_df["train_MAE"]

# % better than the baseline for the same stat (positive = model beats baseline)
baseline_mae = results_df[results_df["model"] == "baseline"].set_index("target")["val_MAE"]
results_df["vs_baseline_%"] = (1 - results_df["val_MAE"] / results_df["target"].map(baseline_mae)) * 100

# NEW: full table for each stat (baseline row included), best val MAE on top
for target in run_targets:
    print(f"\n===== {target} =====")
    table = results_df[results_df["target"] == target].sort_values("val_MAE")
    print(table[["model", "train_MAE", "val_MAE", "gap", "vs_baseline_%"]].round(3).to_string(index=False))

# summary: one row per stat, one column per model (% better than baseline)
print("\n===== % better than baseline =====")
print(results_df.pivot(index="target", columns="model", values="vs_baseline_%").round(2))


FGA | linear: {}
   train MAE = 2.735 | val MAE = 2.812 | gap = 0.077
FGA | ridge: {'ridge__alpha': 100}
   train MAE = 2.737 | val MAE = 2.809 | gap = 0.072
FGA | random_forest: {'max_features': 0.5, 'min_samples_leaf': 50}
   train MAE = 2.600 | val MAE = 2.830 | gap = 0.230
FGA | gradient_boosting: {'learning_rate': 0.03, 'min_samples_leaf': 100}
   train MAE = 2.623 | val MAE = 2.837 | gap = 0.215
FGM | linear: {}
   train MAE = 1.760 | val MAE = 1.775 | gap = 0.015
FGM | ridge: {'ridge__alpha': 100}
   train MAE = 1.760 | val MAE = 1.771 | gap = 0.011
FGM | random_forest: {'max_features': 0.5, 'min_samples_leaf': 100}
   train MAE = 1.719 | val MAE = 1.777 | gap = 0.057
FGM | gradient_boosting: {'learning_rate': 0.03, 'min_samples_leaf': 100}
   train MAE = 1.682 | val MAE = 1.782 | gap = 0.100
PTS | linear: {}
   train MAE = 4.557 | val MAE = 4.650 | gap = 0.093
PTS | ridge: {'ridge__alpha': 100}
   train MAE = 4.558 | val MAE = 4.644 | gap = 0.086
PTS | random_forest: {'max_feat

In [15]:
# NEW: final one-time evaluation on the test set (only run this once!)

# for each stat, pick the model with the lowest validation MAE (can be the baseline)
best_rows = results_df.loc[results_df.groupby("target")["val_MAE"].idxmin()]

X_test = test[features_cols]

test_results = []
for target, name, val_mae in zip(best_rows["target"], best_rows["model"], best_rows["val_MAE"]):
    y_test = test[target]

    # baseline = the stat's own last 5 average, same as before
    baseline_preds = test[target + "_LAST5"]

    # use the saved model, or the baseline if the baseline won on validation
    if name == "baseline":
        preds = baseline_preds
    else:
        preds = best_models[(target, name)].predict(X_test)

    test_results.append({"target": target, "model": name,
                         "val_MAE": val_mae,
                         "test_MAE": mean_absolute_error(y_test, preds),
                         "baseline_test_MAE": mean_absolute_error(y_test, baseline_preds)})

test_df = pd.DataFrame(test_results)

# % better than the baseline on the test set (positive = model beats baseline)
test_df["vs_baseline_%"] = (1 - test_df["test_MAE"] / test_df["baseline_test_MAE"]) * 100

print(test_df.sort_values("vs_baseline_%", ascending=False).round(3).to_string(index=False))


target    model  val_MAE  test_MAE  baseline_test_MAE  vs_baseline_%
   TOV    ridge    0.892     0.914              0.958          4.608
   REB    ridge    1.915     1.925              2.014          4.394
   FTM    ridge    1.341     1.354              1.411          4.045
   FGM    ridge    1.771     1.834              1.910          3.941
   PTS    ridge    4.644     4.813              5.009          3.909
  FG3M    ridge    0.912     0.910              0.945          3.653
   AST   linear    1.352     1.433              1.486          3.519
   STL   linear    0.727     0.731              0.748          2.326
   FGA    ridge    2.809     2.946              3.007          2.040
   BLK baseline    0.532     0.521              0.521          0.000
